# pandas 2/3 : Conditions, masques et indexation (`loc` / `iloc`)

**Objectifs** : filtrer les lignes d'un DataFrame selon des conditions (masques booléens), puis sélectionner précisément des lignes et colonnes avec `loc` (par étiquettes) et `iloc` (par positions).

*Durée estimée : 45 min.*

In [ ]:
# Cellule de préparation : à exécuter en premier (ne fait rien en local)
import os, sys
if "google.colab" in sys.modules and not os.path.exists("intro-data-python"):
    !git clone -q https://github.com/cdrutinuspro/intro-data-python.git
    %cd intro-data-python/notebooks

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/titanic.csv", index_col="PassengerId")
df.head()

Ici, on a utilisé `PassengerId` comme **index** : les étiquettes des lignes vont de 1 à 891, alors que leurs positions vont de 0 à 890. Cette différence est le cœur de la distinction `loc` / `iloc`.

## 1. Conditions et masques

C'est exactement le même principe qu'avec NumPy : une comparaison produit un **masque** de booléens, que l'on utilise pour filtrer.

In [ ]:
masque = df["Age"] < 18
print(masque.head())
print("Nombre de mineurs :", masque.sum())

In [ ]:
mineurs = df[masque]          # ou directement : df[df["Age"] < 18]
mineurs.head()

### Combiner des conditions

Comme avec NumPy : `&` (et), `|` (ou), `~` (non), avec **des parenthèses** autour de chaque condition.

In [ ]:
# Femmes de 1ère classe
femmes_1 = df[(df["Sex"] == "female") & (df["Pclass"] == 1)]
print(len(femmes_1))

# Enfants ou personnes de plus de 70 ans
extremes = df[(df["Age"] < 10) | (df["Age"] > 70)]
print(len(extremes))

# Tous sauf les hommes
print(len(df[~(df["Sex"] == "male")]))

### Outils pratiques pour construire des masques

| Méthode | Rôle |
|---|---|
| `isin([...])` | la valeur appartient à une liste |
| `between(a, b)` | valeur comprise entre a et b (inclus) |
| `isna()` / `notna()` | valeur manquante / présente |
| `str.contains("txt")` | une chaîne contient un motif |

In [ ]:
print(len(df[df["Embarked"].isin(["C", "Q"])]))
print(len(df[df["Age"].between(20, 30)]))
print(len(df[df["Age"].isna()]))
print(len(df[df["Name"].str.contains("Dr.", regex=False)]))

⚠️ Un `NaN` n'est **jamais** vrai pour une comparaison : `df["Age"] < 18` vaut `False` pour les âges inconnus. Ces passagers n'apparaîtront donc ni dans `Age < 18`, ni dans `Age >= 18`.

In [ ]:
print((df["Age"] < 18).sum() + (df["Age"] >= 18).sum(), "sur", len(df))

### Calculer sur un sous-ensemble

On enchaîne : filtrer, choisir une colonne, agréger.

In [ ]:
print("Survie des femmes :", df[df["Sex"] == "female"]["Survived"].mean())
print("Survie des hommes :", df[df["Sex"] == "male"]["Survived"].mean())

---
### ✏️ Exercices (masques)

### ✏️ Exercice 1

Combien de passagers de 3ème classe ont survécu ?

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
print(len(df[(df["Pclass"] == 3) & (df["Survived"] == 1)]))
```

</details>

### ✏️ Exercice 2

Quel est le tarif moyen payé par les passagers qui ont embarqué à Cherbourg (`C`) ?

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
print(df[df["Embarked"] == "C"]["Fare"].mean())
```

</details>

### ✏️ Exercice 3

Quelle proportion des enfants de moins de 12 ans a survécu ? Comparez avec les adultes (12 ans et plus).

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
print(df[df["Age"] < 12]["Survived"].mean())
print(df[df["Age"] >= 12]["Survived"].mean())
```

</details>

## 2. `loc` : sélection par **étiquettes**

Syntaxe : `df.loc[lignes, colonnes]`. Les bornes d'un slice `loc` sont **incluses**.

In [ ]:
print(df.loc[1])                       # la ligne dont l'index vaut 1 (une Series)

In [ ]:
df.loc[1:3]                             # lignes d'index 1, 2 ET 3

In [ ]:
print(df.loc[1:3, "Name"])                      # une colonne -> Series
df.loc[1:3, ["Name", "Age"]]                      # plusieurs colonnes -> DataFrame

In [ ]:
df.loc[1:3, "Name":"Age"]                    # slice de colonnes (bornes incluses aussi)

## 3. `iloc` : sélection par **positions**

Syntaxe : `df.iloc[lignes, colonnes]`, avec des entiers comme pour NumPy (0 = première ligne). Les bornes de fin sont **exclues**.

In [ ]:
print(df.iloc[0])          # la première ligne, quel que soit son index
df.iloc[0:3]               # positions 0, 1 et 2 (3 exclu)

In [ ]:
df.iloc[:3, [2, 3, 4]]      # 3 premières lignes, colonnes en positions 2, 3 et 4


In [ ]:
df.iloc[-3:]               # 3 dernières lignes

### Comparaison directe

Sur nos données, l'étiquette `10` correspond à la position `9`. Observez la différence :

In [ ]:
print(df.loc[10:12, "Name"])    # étiquettes 10, 11, 12
print()
print(df.iloc[10:12]["Name"])    # positions 10 et 11 -> étiquettes 11 et 12

| | `loc` | `iloc` |
|---|---|---|
| se base sur | **étiquettes** | **positions** |
| bornes d'un slice | incluses | fin exclue |
| accepte un masque booléen | oui | pas directement (utiliser `.values`) |

## 4. `loc` avec un masque

C'est l'usage le plus fréquent : choisir **les lignes par une condition** et **les colonnes par leur nom**.

In [ ]:
df.loc[df["Age"] < 5, ["Name", "Age", "Survived"]]

### Modifier des valeurs

Pour **modifier** un sous-ensemble, on utilise `loc` sur le DataFrame d'origine. Ne tentez pas `df[df["Age"] < 5]["Survived"] = ...` : cela travaille sur une copie temporaire et ne modifie pas `df`.

In [ ]:
df2 = df.copy()

# Remplacer les âges manquants par la médiane
mediane = df2["Age"].median()
df2.loc[df2["Age"].isna(), "Age"] = mediane
print(df2["Age"].isna().sum())

# Créer une colonne catégorielle à partir d'une condition
df2["Mineur"] = False
df2.loc[df2["Age"] < 18, "Mineur"] = True
df2["Mineur"].value_counts()

## 5. Changer l'index : `set_index` / `reset_index`

Un index porteur de sens facilite l'accès avec `loc`.

In [ ]:
par_nom = df.set_index("Name")
par_nom.loc["Braund, Mr. Owen Harris", ["Age", "Fare"]]

In [ ]:
par_nom.reset_index().head(3)    # on remet un index 0, 1, 2... et Name redevient colonne

## Récapitulatif

| Je veux… | Code |
|---|---|
| filtrer les lignes | `df[condition]` |
| combiner des conditions | `(c1) & (c2)`, `(c1) \| (c2)`, `~(c)` |
| appartenance, intervalle, manquant | `isin`, `between`, `isna` |
| par étiquettes | `df.loc[lignes, colonnes]` |
| par positions | `df.iloc[lignes, colonnes]` |
| filtrer + choisir des colonnes | `df.loc[condition, ["c1", "c2"]]` |
| modifier une sélection | `df.loc[condition, "col"] = valeur` |

---
## Exercices (`loc` / `iloc`)

### ✏️ Exercice 4

Avec `iloc`, affichez le nom et l'âge des 5 derniers passagers du fichier. *Indice : `Name` est en position 2 et `Age` en position 4.*

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
df.iloc[-5:, [2, 4]]
```

</details>

### ✏️ Exercice 5

Avec `loc`, affichez les noms et classes (`Pclass`) des passagers d'index 100 à 105 (inclus).

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
df.loc[100:105, ["Name", "Pclass"]]
```

</details>

### ✏️ Exercice 6

Avec `loc` et un masque, affichez nom, âge et tarif des passagers de 1ère classe ayant payé plus de 200.

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
df.loc[(df["Pclass"] == 1) & (df["Fare"] > 200), ["Name", "Age", "Fare"]]
```

</details>

### ✏️ Exercice 7

Dans une copie de `df`, remplacez les valeurs manquantes de `Embarked` par `"S"` (le port le plus fréquent). Vérifiez qu'il ne reste plus de manquants dans cette colonne.

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
d = df.copy()
d.loc[d["Embarked"].isna(), "Embarked"] = "S"
print(d["Embarked"].isna().sum())
```

</details>

### ✏️ Exercice 8

**Défi** : parmi les passagers dont le nom contient `"Mrs."`, quelle proportion a survécu ? Comparez avec ceux dont le nom contient `"Miss."`.

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
print(df[df["Name"].str.contains("Mrs.", regex=False)]["Survived"].mean())
print(df[df["Name"].str.contains("Miss.", regex=False)]["Survived"].mean())
```

</details>